# Recommender — Data Preparation (Option A: latest month snapshot)

Goal: reduce the sampled Santander dataset to one row per customer (their most recent product-holding state), then build the customer-product matrix used as input for collaborative filtering.

See `README.md` in this folder for the full reasoning behind these choices.

## 1. Load and translate columns

In [1]:
import pandas as pd
import numpy as np
from columns import COLUMN_MAPPING, PRODUCT_COLUMNS

df = pd.read_csv("../../data/santander/train_sampled_75k.csv", low_memory=False)
df = df.rename(columns=COLUMN_MAPPING)

df.shape

(1069246, 48)

In [2]:
df[["customer_id", "snapshot_date"] + PRODUCT_COLUMNS].head()

,customer_id,snapshot_date,product_saving_account,product_guarantees,product_current_account,product_derivada_account,product_payroll_account,product_junior_account,product_mas_particular_account,product_particular_account,...,product_mortgage,product_pension_plan,product_loans,product_taxes,product_credit_card,product_securities,product_home_account,product_payroll,product_pension_payment,product_direct_debit
0,1375586,2015-01-28,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,0
1,1050592,2015-01-28,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,0
2,1050599,2015-01-28,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,0
3,1050676,2015-01-28,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,0
4,1050706,2015-01-28,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,0


## 2. Check the date column

`snapshot_date` is loaded as plain text by default -- we need it as an actual date type so pandas can correctly identify which row is "most recent" per customer (comparing text like "2015-9-28" vs "2015-10-28" alphabetically would give the wrong answer).

In [3]:
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
df["snapshot_date"].dtype

dtype('<M8[us]')

In [4]:
df["snapshot_date"].min(), df["snapshot_date"].max()

(Timestamp('2015-01-28 00:00:00'), Timestamp('2016-05-28 00:00:00'))

**Check:** the min/max should span the full period the dataset documentation describes (17 months). If the range looks wrong (e.g. only a few days), the date parsing likely failed silently and needs a `format=` argument.

## 3. Reduce to the latest month per customer

Sort by date, group by customer, keep only the last row of each group.

In [5]:
df_latest = (
    df.sort_values("snapshot_date")
      .groupby("customer_id", as_index=False)
      .tail(1)
)

df_latest.shape

(75000, 48)

**Expected result:** `(75000, N)` -- one row per sampled customer. If the row count is higher, some customers have two rows tied on the same latest date (a known edge case in this dataset for a small number of customers); if lower, some customer_ids may not have parsed correctly.

In [6]:
df_latest["customer_id"].duplicated().sum()  # should be 0

np.int64(0)

## 4. Handle missing values in product columns

A known quirk of this dataset: a handful of rows have `NaN` in some product columns instead of 0, usually for customers with incomplete early history. Since a product column only ever means "holds" (1) or "does not hold" (0), a missing value here safely means "does not hold" -- there's no ambiguous in-between state for these binary indicators.

In [7]:
df_latest[PRODUCT_COLUMNS].isnull().sum().sum()  # total missing cells across all product columns

np.int64(954)

In [8]:
df_latest[PRODUCT_COLUMNS] = df_latest[PRODUCT_COLUMNS].fillna(0).astype(int)

## 5. Build the customer-product matrix

This dataset is already in *wide* format (one column per product) -- no pivot needed. The matrix is simply the product columns, indexed by `customer_id`.

In [9]:
product_matrix = df_latest.set_index("customer_id")[PRODUCT_COLUMNS]
product_matrix.shape

(75000, 24)

In [10]:
product_matrix.head()

,product_saving_account,product_guarantees,product_current_account,product_derivada_account,product_payroll_account,product_junior_account,product_mas_particular_account,product_particular_account,product_particular_plus_account,product_short_term_deposit,...,product_mortgage,product_pension_plan,product_loans,product_taxes,product_credit_card,product_securities,product_home_account,product_payroll,product_pension_payment,product_direct_debit
customer_id,,,,,,,,,,,,,,,,,,,,,
196769,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
154362,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
211954,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
204184,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
226031,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 6. Sanity checks before moving to modeling

In [11]:
# Matrix density: what fraction of (customer, product) cells are 1?
# Useful reference point -- collaborative filtering tends to work better
# with denser matrices; a very sparse matrix (most cells 0) is normal for
# this kind of data, but worth knowing the baseline.
density = product_matrix.values.mean()
print(f"Matrix density: {density:.2%}")

Matrix density: 5.41%


In [12]:
# How many products does the average customer hold?
product_matrix.sum(axis=1).describe()

count    75000.000000
mean         1.297240
std          1.471021
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max         12.000000
dtype: float64

In [13]:
# Which products are most commonly held? (useful later to sanity-check
# recommendations -- a model that only ever recommends the single most
# popular product isn't learning much)
product_matrix.sum(axis=0).sort_values(ascending=False)

product_current_account            44308
product_direct_debit                8879
product_particular_account          7771
product_e_account                   5810
product_payroll_account             5640
product_pension_payment             4093
product_taxes                       3758
product_payroll                     3731
product_credit_card                 2717
product_particular_plus_account     2584
product_long_term_deposit           2508
product_securities                  1636
product_funds                       1151
product_mas_particular_account       644
product_junior_account               608
product_pension_plan                 571
product_mortgage                     340
product_home_account                 256
product_loans                        157
product_medium_term_deposit           79
product_short_term_deposit            25
product_derivada_account              22
product_saving_account                 3
product_guarantees                     2
dtype: int64

Train the collaborative filtering model on `product_matrix` and evaluate with precision@k / recall@k.

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# product_matrix : 75000 clients x 24 produits, déjà construite
similarity_matrix = cosine_similarity(product_matrix.T)  # .T : on compare les produits (colonnes), pas les clients (lignes)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=product_matrix.columns,
    columns=product_matrix.columns
)

similarity_df.round(2)

,product_saving_account,product_guarantees,product_current_account,product_derivada_account,product_payroll_account,product_junior_account,product_mas_particular_account,product_particular_account,product_particular_plus_account,product_short_term_deposit,...,product_mortgage,product_pension_plan,product_loans,product_taxes,product_credit_card,product_securities,product_home_account,product_payroll,product_pension_payment,product_direct_debit
product_saving_account,1.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.00,0.00,0.01
product_guarantees,0.00,1.00,0.01,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.02,0.00,0.00,0.00,0.00,0.00,0.02
product_current_account,0.00,0.01,1.00,0.02,0.03,0.0,0.07,0.28,0.14,0.02,...,0.03,0.06,0.02,0.13,0.11,0.14,0.05,0.04,0.04,0.25
product_derivada_account,0.00,0.00,0.02,1.00,0.02,0.0,0.00,0.01,0.01,0.00,...,0.01,0.01,0.02,0.03,0.02,0.04,0.00,0.02,0.02,0.02
product_payroll_account,0.00,0.00,0.03,0.02,1.00,0.0,0.04,0.11,0.21,0.01,...,0.15,0.13,0.02,0.35,0.40,0.16,0.05,0.75,0.79,0.57
product_junior_account,0.00,0.00,0.00,0.00,0.00,1.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
product_mas_particular_account,0.00,0.00,0.07,0.00,0.04,0.0,1.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.03,0.03,0.01,0.00,0.03,0.03,0.06
product_particular_account,0.00,0.00,0.28,0.01,0.11,0.0,0.00,1.00,0.11,0.02,...,0.04,0.08,0.01,0.15,0.12,0.14,0.09,0.10,0.11,0.15
product_particular_plus_account,0.00,0.00,0.14,0.01,0.21,0.0,0.00,0.11,1.00,0.02,...,0.09,0.08,0.01,0.19,0.19,0.13,0.06,0.18,0.20,0.23
product_short_term_deposit,0.00,0.00,0.02,0.00,0.01,0.0,0.00,0.02,0.02,1.00,...,0.00,0.00,0.00,0.02,0.01,0.02,0.00,0.01,0.01,0.01


In [15]:
import joblib

joblib.dump(similarity_df, "../recommender_similarity.joblib")
joblib.dump(product_matrix, "../recommender_product_matrix.joblib")

['../recommender_product_matrix.joblib']